# Run Full Pipeline + Visualize Results

This notebook runs the actual filtering job and generates the final artifacts needed for the submission:
- per-sample metrics
- filtering decisions
- keep/review/reject manifests
- summary JSON
- plots for visual analysis


In [ ]:
%%capture
!pip install -q polars pandas matplotlib plotly


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/indic_audio_filter"
PROJECT_DIR = f"{DRIVE_DIR}/project_repo"
DATA_DIR = f"{DRIVE_DIR}/data"
RUN_DIR = f"{DRIVE_DIR}/outputs/run_v1"

import os, sys, glob, subprocess, json
os.makedirs(RUN_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)


In [ ]:
manifest_dirs = sorted(glob.glob(f"{DATA_DIR}/manifests/*_manifests"))
manifest_dirs[:5]


In [ ]:
one_lang = manifest_dirs[0]
cmd = [
    "python", "-m", "src.main",
    "--manifest", one_lang,
    "--config", f"{PROJECT_DIR}/configs/default.yaml",
    "--output_dir", f"{RUN_DIR}/single_language"
]
res = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
print("return code:", res.returncode)
print(res.stdout[-2000:])
print(res.stderr[-2000:])


In [ ]:
cmd = [
    "python", "-m", "src.main",
    "--manifest", f"{DATA_DIR}/manifests",
    "--config", f"{PROJECT_DIR}/configs/default.yaml",
    "--output_dir", RUN_DIR
]
res = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
print("return code:", res.returncode)
print(res.stdout[-5000:])
print(res.stderr[-3000:])


In [ ]:
import json, polars as pl
with open(f"{RUN_DIR}/summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
summary


In [ ]:
df = pl.read_parquet(f"{RUN_DIR}/metrics.parquet")
df.head(10)


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

for name in sorted(os.listdir(f"{RUN_DIR}/plots")):
    img = Image.open(os.path.join(f"{RUN_DIR}/plots", name))
    plt.figure(figsize=(8,4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(name)
    plt.show()


## Recommended outputs to highlight in your final report

- total kept / reviewed / rejected
- keep rate by language
- score distribution
- example rejection reasons
- examples from the review bucket

That gives the submission both **engineering depth** and **visual clarity**.
